# 03 Eval Subset

Runnable notebook for locking a fixed 20–30 clip subset and executing a lightweight subset run.

- Uses the same bootstrap path as D1/custom video: `bash setup_colab.sh --with-models`
- Builds `subset_manifest.csv` from a Drive source folder
- Expects one primary tag per selected clip: `easy`, `occlusion`, `small_object`, or `crowded`


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
REPO_URL = 'https://github.com/ChiCoTheLaAnh/ProjectFinalCS415.git'
REPO_DIR = '/content/ProjectFinalCS415'

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"


In [ ]:
%cd /content/ProjectFinalCS415
!bash setup_colab.sh --with-models


In [ ]:
import shutil
import subprocess
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Torch CUDA runtime:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())

nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is not None:
    gpu_name = subprocess.run(
        [nvidia_smi, '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
    print('GPU:', gpu_name)

if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required for subset evaluation.')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/cv-final-project')
SOURCE_VIDEO_DIR = DRIVE_ROOT / 'inputs' / 'subset_candidates'
MANIFEST_PATH = Path('/content/ProjectFinalCS415/data/processed/subset_manifest.csv')
OUTPUT_ROOT = DRIVE_ROOT / 'results' / 'quantitative' / 'subset_eval'
GROUNDING_CKPT = DRIVE_ROOT / 'checkpoints' / 'groundingdino_swint_ogc.pth'
SAM2_CKPT = DRIVE_ROOT / 'checkpoints' / 'sam2.1_hiera_small.pt'
SUBSET_LIMIT = None
DEVICE = 'cuda'

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Source folder:', SOURCE_VIDEO_DIR)
print('Manifest path:', MANIFEST_PATH)
print('Output root:', OUTPUT_ROOT)


In [ ]:
from pathlib import Path

for required_path in (SOURCE_VIDEO_DIR, GROUNDING_CKPT, SAM2_CKPT):
    if not Path(required_path).exists():
        raise FileNotFoundError(f'Missing required path: {required_path}')
    print(required_path)


In [ ]:
%cd /content/ProjectFinalCS415
from src.data.build_subset import build_subset_manifest

manifest_info = build_subset_manifest(SOURCE_VIDEO_DIR, MANIFEST_PATH)
manifest_info


## Manual Review Step

Open `data/processed/subset_manifest.csv` and finalize the first fixed subset.

Rules for this round:

- mark exactly 20–30 rows with `selected = 1`
- assign one `primary_tag` per selected clip from:
  - `easy`
  - `occlusion`
  - `small_object`
  - `crowded`
- optionally add short notes per clip


In [ ]:
import pandas as pd

manifest_df = pd.read_csv(MANIFEST_PATH)
display(manifest_df.head(20))
print('Selected rows:', int((manifest_df['selected'].astype(str) == '1').sum()))
print('Primary tag counts (non-empty):')
print(manifest_df[manifest_df['primary_tag'].fillna('') != '']['primary_tag'].value_counts())


In [ ]:
selected_df = pd.read_csv(MANIFEST_PATH)
selected_df['selected'] = selected_df['selected'].astype(str)
chosen = selected_df[selected_df['selected'] == '1']
if not 20 <= len(chosen) <= 30:
    raise ValueError(f'Expected 20–30 selected clips, found {len(chosen)}')
invalid_tags = sorted(set(chosen['primary_tag']) - {'easy', 'occlusion', 'small_object', 'crowded'})
if invalid_tags:
    raise ValueError(f'Invalid primary_tag values: {invalid_tags}')
display(chosen[['clip_id', 'video_path', 'primary_tag', 'notes']].head(30))
print(chosen['primary_tag'].value_counts())


In [ ]:
%cd /content/ProjectFinalCS415
import subprocess

cmd = [
    'python3', 'scripts/run_eval_subset.py',
    '--config', 'configs/base.yaml',
    '--manifest', str(MANIFEST_PATH),
    '--output_dir', str(OUTPUT_ROOT),
    '--grounding_ckpt', str(GROUNDING_CKPT),
    '--sam2_ckpt', str(SAM2_CKPT),
    '--device', DEVICE,
]
if SUBSET_LIMIT is not None:
    cmd.extend(['--limit', str(SUBSET_LIMIT)])
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
from pathlib import Path

summary_path = OUTPUT_ROOT / 'subset_run_summary.json'
if not summary_path.exists():
    raise FileNotFoundError(f'Missing subset run summary: {summary_path}')
summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, indent=2))
print('Tag counts:', summary.get('tag_counts', {}))
print('Completed clips:', summary.get('num_completed'))
